# Training ConvNeXt-Tiny for Bee Orientation Regression

This notebook implements a complete PyTorch pipeline to train a **ConvNeXt-Tiny** model to regress the orientation angle of cropped bee images.

### Key Approach:

- **Circular Targets:** Rather than regressing raw angles directly (which introduces discontinuity at 0/360 degrees), we project angles to the unit circle: $\mathbf{y} = [\sin(\alpha), \cos(\alpha)]$.
- **Evaluation:** Predictions are mapped back to degrees using `atan2` and validated with a wrap-around adjusted Mean Absolute Error (MAE).


### 1. Imports

Load core machine learning frameworks, data processing utilities, model architectures, and plotting tools.


In [ ]:
import os
from pathlib import Path
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.models import convnext_tiny, ConvNeXt_Tiny_Weights
from PIL import Image
from tqdm import tqdm
import matplotlib.pyplot as plt

### 2. Environment and Paths Setup

Configure execution device (CUDA GPU if available, otherwise CPU) and set path locations for trajectory files and image crops.


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

data_dir = Path("./data")
crops_dir = data_dir / "crops"
trajectory_dir = data_dir / "rec1_trajectories"

### 3. Parse Trajectory Ground Truths

Load and read trajectory `.txt` files to construct a mapping from unique `(trajectory_id, frame_id)` combinations to their ground truth orientation angle (stored in degrees in the 5th column).


In [ ]:
print("Loading trajectories and building orientation angle lookup...")
angle_lookup = {}

for txt_file in sorted(list(trajectory_dir.glob("*.txt"))):
    tid = txt_file.stem
    with open(txt_file, "r") as f:
        for line in f:
            parts = line.strip().split(",")
            if len(parts) >= 5:
                frame_id = int(parts[0])
                angle = float(parts[4])
                angle_lookup[(tid, frame_id)] = angle

print(f"Loaded {len(angle_lookup)} coordinates from trajectory files.")

### 4. Custom Dataset Definition

Define a custom PyTorch `Dataset` class that maps image crop paths, reads files, and calculates the sine and cosine targets for orientation.


In [ ]:
class BeeOrientationDataset(Dataset):
    def __init__(self, split_dir, angle_lookup, transform=None):
        self.split_dir = Path(split_dir)
        self.transform = transform
        self.angle_lookup = angle_lookup
        self.samples = []

        # Iterate directory structure: crops/<split>/<trajectory_id>/frame_xxxxxx.png
        for traj_dir in self.split_dir.iterdir():
            if traj_dir.is_dir():
                tid = traj_dir.name
                for img_path in traj_dir.glob("*.png"):
                    try:
                        frame_id = int(img_path.stem.split("_")[1])
                        if (tid, frame_id) in self.angle_lookup:
                            self.samples.append((img_path, tid, frame_id))
                    except IndexError, ValueError:
                        continue

        print(f"Loaded {len(self.samples)} samples from {self.split_dir.name} split.")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, tid, frame_id = self.samples[idx]
        image = Image.open(img_path).convert("RGB")

        # Transform degrees -> radians -> sin/cos targets
        angle_deg = self.angle_lookup[(tid, frame_id)]
        angle_rad = np.deg2rad(angle_deg)
        target = torch.tensor(
            [np.sin(angle_rad), np.cos(angle_rad)], dtype=torch.float32
        )

        if self.transform:
            image = self.transform(image)

        return image, target

### 5. Augmentations and DataLoaders

Prepare PyTorch normalization and resizing transforms (resized to $224 \times 224$ pixels to fit standard pretrained ConvNeXt-Tiny settings) and set up the corresponding DataLoaders.


In [ ]:
train_transform = transforms.Compose(
    [
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ]
)

val_transform = transforms.Compose(
    [
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ]
)

train_dataset = BeeOrientationDataset(
    crops_dir / "train", angle_lookup, transform=train_transform
)
val_dataset = BeeOrientationDataset(
    crops_dir / "val", angle_lookup, transform=val_transform
)
test_dataset = BeeOrientationDataset(
    crops_dir / "test", angle_lookup, transform=val_transform
)

batch_size = 32
train_loader = DataLoader(
    train_dataset, batch_size=batch_size, shuffle=True, num_workers=4, pin_memory=True
)
val_loader = DataLoader(
    val_dataset, batch_size=batch_size, shuffle=False, num_workers=4, pin_memory=True
)
test_loader = DataLoader(
    test_dataset, batch_size=batch_size, shuffle=False, num_workers=4, pin_memory=True
)

### 6. Model Initialization

Load ConvNeXt-Tiny with ImageNet weights and swap out its final classifier projection layer to map features to the 2 orientation outputs (sine and cosine projections).


In [ ]:
print("Initializing ConvNeXt-Tiny model...")
model = convnext_tiny(weights=ConvNeXt_Tiny_Weights.DEFAULT)

# Output size 2 for (sin, cos)
in_features = model.classifier[2].in_features
model.classifier[2] = nn.Linear(in_features, 2)

model = model.to(device)
print(model.classifier)

### 7. Loss, Optimizer, and Scheduler

Configure optimization utilities: MSE loss on the sine and cosine parameters, the AdamW optimizer, and Cosine Annealing learning rate schedule.


In [ ]:
criterion = nn.MSELoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-2)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=10)

### 8. Custom Angular Metric

Implement the Mean Absolute Error (MAE) metric adjusted for circular wrapping to accurately score prediction error in degrees.


In [ ]:
def compute_angular_error_deg(pred, target):
    """Computes Mean Absolute Error (MAE) in degrees using circular wrapping."""
    pred_rad = torch.atan2(pred[:, 0], pred[:, 1])
    target_rad = torch.atan2(target[:, 0], target[:, 1])

    diff_rad = pred_rad - target_rad
    diff_rad = torch.atan2(torch.sin(diff_rad), torch.cos(diff_rad))

    diff_deg = torch.abs(torch.rad2deg(diff_rad))
    return diff_deg.mean().item()

### 9. Training and Validation Step Helpers

Write loop functions to run training and evaluation epochs and log losses and angular MAEs.


In [ ]:
def train_epoch(model, loader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0
    running_mae = 0.0

    for images, targets in tqdm(loader, desc="Training", leave=False):
        images, targets = images.to(device), targets.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, targets)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        running_mae += compute_angular_error_deg(outputs, targets) * images.size(0)

    epoch_loss = running_loss / len(loader.dataset)
    epoch_mae = running_mae / len(loader.dataset)
    return epoch_loss, epoch_mae


def validate(model, loader, criterion, device):
    model.eval()
    running_loss = 0.0
    running_mae = 0.0

    with torch.no_grad():
        for images, targets in tqdm(loader, desc="Validating", leave=False):
            images, targets = images.to(device), targets.to(device)

            outputs = model(images)
            loss = criterion(outputs, targets)

            running_loss += loss.item() * images.size(0)
            running_mae += compute_angular_error_deg(outputs, targets) * images.size(0)

    val_loss = running_loss / len(loader.dataset)
    val_mae = running_mae / len(loader.dataset)
    return val_loss, val_mae

### 10. Main Training Loop

Execute training across epochs, update validation performance logs, and cache the best checkpoint on disk.


In [ ]:
num_epochs = 10
best_val_loss = float("inf")
history = {"train_loss": [], "train_mae": [], "val_loss": [], "val_mae": []}

print("Starting training...")
for epoch in range(num_epochs):
    train_loss, train_mae = train_epoch(
        model, train_loader, criterion, optimizer, device
    )
    val_loss, val_mae = validate(model, val_loader, criterion, device)
    scheduler.step()

    history["train_loss"].append(train_loss)
    history["train_mae"].append(train_mae)
    history["val_loss"].append(val_loss)
    history["val_mae"].append(val_mae)

    print(f"Epoch {epoch + 1}/{num_epochs}:")
    print(f"  Train Loss (MSE): {train_loss:.4f} | Train MAE: {train_mae:.2f}°")
    print(f"  Val Loss (MSE): {val_loss:.4f} | Val MAE: {val_mae:.2f}°")

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), "best_convnext_tiny_bee_orientation.pth")
        print("  --> Saved best model checkpoint!")

### 11. Plot Training History

Plot trends for training/validation losses and angular MAEs to evaluate model optimization and trace convergence.


In [ ]:
epochs = range(1, len(history["train_loss"]) + 1)
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(epochs, history["train_loss"], label="Train Loss")
plt.plot(epochs, history["val_loss"], label="Val Loss")
plt.title("MSE Loss vs Epochs")
plt.xlabel("Epochs")
plt.ylabel("Loss")
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(epochs, history["train_mae"], label="Train MAE (°)")
plt.plot(epochs, history["val_mae"], label="Val MAE (°)")
plt.title("Mean Absolute Angular Error vs Epochs")
plt.xlabel("Epochs")
plt.ylabel("MAE (Degrees)")
plt.legend()

plt.tight_layout()
plt.show()

### 12. Evaluate on Test Split

Load the saved model parameters that achieved the lowest validation loss, run evaluation on the test DataLoader, and report final metrics.


In [ ]:
model.load_state_dict(torch.load("best_convnext_tiny_bee_orientation.pth"))
test_loss, test_mae = validate(model, test_loader, criterion, device)
print(f"\nFinal Test Results:")
print(f"  Test Loss (MSE): {test_loss:.4f}")
print(f"  Test MAE: {test_mae:.2f}°")